<a href="https://colab.research.google.com/github/sampleyeon/SAS-data-analysis/blob/main/submission2(3%EC%B0%A8_%EC%A0%9C%EC%B6%9C).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


1. 데이터 로드

In [ ]:
customer = pd.read_csv("/content/drive/MyDrive/2026 공모전-SAS 데이터분석/2회 경진대회 데이터/train/train_customer_info.csv")
transaction = pd.read_csv("/content/drive/MyDrive/2026 공모전-SAS 데이터분석/2회 경진대회 데이터/train/train_transaction_history.csv")
finance = pd.read_csv("/content/drive/MyDrive/2026 공모전-SAS 데이터분석/2회 경진대회 데이터/train/train_finance_profile.csv")
target = pd.read_csv("/content/drive/MyDrive/2026 공모전-SAS 데이터분석/2회 경진대회 데이터/train/train_targets.csv")

- shape, 컬럼 확인

In [ ]:
datasets = {
    'customer'    : customer,
    'transaction' : transaction,
    'finance'     : finance,
    'target'      : target
}

for name, df in datasets.items():
    print(f"\n{'='*45}")
    print(f"  [{name}]  {df.shape[0]:,}행 × {df.shape[1]}열")
    print(f"{'='*45}")
    print(df.dtypes)


  [customer]  60,000행 × 8열
customer_id        object
join_date          object
age                 int64
gender             object
region_code        object
is_married          int64
prefer_category    object
income_group       object
dtype: object

  [transaction]  1,048,575행 × 7열
trans_id          object
customer_id       object
trans_date        object
trans_amount       int64
biz_type          object
item_category     object
is_installment     int64
dtype: object

  [finance]  60,000행 × 9열
customer_id               object
credit_score               int64
num_active_cards           int64
total_deposit_balance      int64
total_loan_balance         int64
card_cash_service_amt      int64
card_loan_amt              int64
fin_overdue_days           int64
fin_asset_trend_score    float64
dtype: object

  [target]  60,000행 × 3열
customer_id      object
target_churn      int64
target_ltv      float64
dtype: object


- 결측치 확인

In [ ]:
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n[{name}]")
    if missing.empty:
        print("  결측치 없음")
    else:
        pct = (missing / len(df) * 100).round(2)
        print(pd.DataFrame({'결측 수': missing, '비율(%)': pct}))


[customer]
  결측치 없음

[transaction]
  결측치 없음

[finance]
  결측치 없음

[target]
  결측치 없음


- 샘플 확인

In [ ]:
for name, df in datasets.items():
    print(f"\n[{name}]")
    display(df.head(3))


[customer]


,customer_id,join_date,age,gender,region_code,is_married,prefer_category,income_group
0,C000001,2020-04-12,36,F,R03,1,Grocery,G4
1,C000002,2021-03-11,32,F,R02,0,Home,G3
2,C000003,2022-05-10,41,M,R03,1,Fashion,G3



[transaction]


,trans_id,customer_id,trans_date,trans_amount,biz_type,item_category,is_installment
0,T00000001,C068329,2023-07-31,14855,Online,Home,0
1,T00000002,C044011,2023-10-09,40476,Offline,Fashion,0
2,T00000003,C014765,2023-11-24,186238,Online,Home,0



[finance]


,customer_id,credit_score,num_active_cards,total_deposit_balance,total_loan_balance,card_cash_service_amt,card_loan_amt,fin_overdue_days,fin_asset_trend_score
0,C000001,713,6,2517740,58608891,100572,65339,0,0.551107
1,C000002,869,5,679696,33403843,210018,0,0,-0.342776
2,C000003,588,2,17319511,0,4092,0,1,-0.949003



[target]


,customer_id,target_churn,target_ltv
0,C000001,0,556691.0
1,C000002,0,1460203.0
2,C000003,0,605476.0


파악된 내용 요약:

- 결측치 없음 → 별도 처리 불필요

- transaction이 1,048,575행 → 고객 1인당 평균 약 17건

- join_date, trans_date → 날짜 변환 필요

- gender, region_code, prefer_category, income_group, biz_type, item_category → 인코딩 필요

- 타겟 분포 확인

In [ ]:
# ── Churn 분포 ──
churn_vc  = target['target_churn'].value_counts().sort_index()
churn_pct = (churn_vc / len(target) * 100).round(2)

print("[ Churn 분포 ]")
print(pd.DataFrame({'count': churn_vc, '%': churn_pct}))
print(f"\n  → 이탈률: {target['target_churn'].mean():.2%}")

[ Churn 분포 ]
              count      %
target_churn              
0             54071  90.12
1              5929   9.88

  → 이탈률: 9.88%


In [ ]:
# ── LTV 기초통계 & 왜도 ──
ltv = target['target_ltv']

print("\n[ LTV 기초통계 ]")
print(ltv.describe().round(0))

skewness = ltv.skew()
print(f"\n  왜도(skewness): {skewness:.2f}")

if skewness > 1:
    print("  → 오른쪽 치우침 심함 ⚠️  log 변환 필수")
elif skewness > 0.5:
    print("  → 약간 치우침, log 변환 권장")
else:
    print("  → 정규분포에 가까움 ✅")

# ── log 변환 후 왜도 비교 ──
ltv_log = np.log1p(ltv)
print(f"  log 변환 후 왜도: {ltv_log.skew():.2f}")


[ LTV 기초통계 ]
count       60000.0
mean      1239290.0
std       1360034.0
min             3.0
25%        257648.0
50%        807154.0
75%       1751954.0
max      17176629.0
Name: target_ltv, dtype: float64

  왜도(skewness): 2.12
  → 오른쪽 치우침 심함 ⚠️  log 변환 필수
  log 변환 후 왜도: -1.14


- transaction 기본 분석

In [ ]:
# 날짜 변환 (아직 안 했을 경우)
transaction['trans_date'] = pd.to_datetime(transaction['trans_date'])

# ── 거래 기간 ──
print("[ 거래 기간 ]")
print(f"  시작: {transaction['trans_date'].min().date()}")
print(f"  종료: {transaction['trans_date'].max().date()}")

# ── 월별 거래 건수 ──
print("\n[ 월별 거래 건수 ]")
transaction['month'] = transaction['trans_date'].dt.to_period('M')
print(transaction.groupby('month').size().to_frame('거래건수'))

[ 거래 기간 ]
  시작: 2023-07-01
  종료: 2023-12-31

[ 월별 거래 건수 ]
           거래건수
month          
2023-07  176934
2023-08  177007
2023-09  170360
2023-10  177448
2023-11  170588
2023-12  176238


In [ ]:
# ── 고객 1인당 거래 건수 ──
print("[ 고객 1인당 거래 건수 ]")
tx_per_customer = transaction.groupby('customer_id').size()
print(tx_per_customer.describe().round(1))

# ── biz_type / item_category 분포 ──
print("\n[ biz_type 분포 ]")
print(transaction['biz_type'].value_counts())

print("\n[ item_category 분포 ]")
print(transaction['item_category'].value_counts())

[ 고객 1인당 거래 건수 ]
count    60000.0
mean        17.5
std          4.2
min          4.0
25%         15.0
50%         17.0
75%         20.0
max         36.0
dtype: float64

[ biz_type 분포 ]
biz_type
Online     629255
Offline    419320
Name: count, dtype: int64

[ item_category 분포 ]
item_category
Electronics    210044
Grocery        209978
Beauty         209925
Fashion        209797
Home           208831
Name: count, dtype: int64


- finance 이상치 확인

In [ ]:
# 연체일수 분포 (churn 신호 핵심 변수)
print("[ fin_overdue_days ]")
print(finance['fin_overdue_days'].describe().round(1))
print(f"  연체 있는 고객 수: {(finance['fin_overdue_days'] > 0).sum():,}명")

# 신용점수 범위 확인
print("\n[ credit_score ]")
print(finance['credit_score'].describe().round(1))

[ fin_overdue_days ]
count    60000.0
mean         0.4
std          0.6
min          0.0
25%          0.0
50%          0.0
75%          1.0
max          5.0
Name: fin_overdue_days, dtype: float64
  연체 있는 고객 수: 19,875명

[ credit_score ]
count    60000.0
mean       739.5
std        109.2
min        308.0
25%        665.0
50%        740.0
75%        814.0
max       1000.0
Name: credit_score, dtype: float64


**📋 EDA 결과 요약 및 전략 포인트**

| 항목 | 결과 | 전략 |
|------|------|------|
| Churn 비율 | 9.88% (불균형) | `class_weight` 또는 `scale_pos_weight` 적용 |
| LTV 왜도 | 2.12 → log 후 -1.14 | log 변환 필수 / log1p 사용 |
| 거래 기간 | 23.07 ~ 23.12 (6개월) | 월별 추세 feature 생성 가능 |
| 고객당 거래 | 평균 17.5건 (min 4, max 36) | 이상치 없음, 안정적 |
| biz_type | Online 60% / Offline 40% | 온라인 비율 feature화 |
| item_category | 5개 균등 분포 | 카테고리 다양성 feature화 |
| fin_overdue_days | 연체 고객 19,875명 (33%) | churn 핵심 신호 |
| credit_score | 308~1000, 평균 740 | 정상 범위 ✅ |

> **log 변환 후 왜도가 -1.14로** 반대로 치우쳤습니다. `log1p` 대신 `np.sqrt` 또는 `Box-Cox` 변환도 나중에 비교해볼 필요 있습니다.



2. transaction 집계

In [ ]:
# 기준일 설정 (거래 마지막 날 다음날)
reference_date = pd.Timestamp('2024-01-01')

transaction['trans_date'] = pd.to_datetime(transaction['trans_date'])

# ── Recency / Frequency / Monetary ──
agg_base = transaction.groupby('customer_id').agg(
    recency        = ('trans_date', lambda x: (reference_date - x.max()).days),
    frequency      = ('trans_id', 'count'),
    total_amount   = ('trans_amount', 'sum'),
    mean_amount    = ('trans_amount', 'mean'),
    std_amount     = ('trans_amount', 'std'),
    max_amount     = ('trans_amount', 'max'),
    min_amount     = ('trans_amount', 'min'),
).reset_index()

print(agg_base.shape)
agg_base.head(3)

(60000, 8)


,customer_id,recency,frequency,total_amount,mean_amount,std_amount,max_amount,min_amount
0,C000001,5,20,1055529,52776.450000,51633.636367,227979,12136
1,C000002,3,14,922154,65868.142857,65206.573884,266110,12019
2,C000003,1,16,812618,50788.625000,42843.526573,166157,13903


In [ ]:
# ── 최근 1개월 vs 이전 소비 비교 (소비 감소 추세 → churn 신호) ──
last_1m = transaction[transaction['trans_date'] >= pd.Timestamp('2023-12-01')]
prev_5m = transaction[transaction['trans_date'] <  pd.Timestamp('2023-12-01')]

last_1m_agg = last_1m.groupby('customer_id')['trans_amount'].sum().rename('amt_last_1m')
prev_5m_agg = prev_5m.groupby('customer_id')['trans_amount'].sum().rename('amt_prev_5m')

agg_trend = pd.concat([last_1m_agg, prev_5m_agg], axis=1).fillna(0)

# 소비 감소율 (음수 = 감소)
agg_trend['amt_trend_ratio'] = (
    agg_trend['amt_last_1m'] - agg_trend['amt_prev_5m'] / 5
) / (agg_trend['amt_prev_5m'] / 5 + 1)

agg_trend = agg_trend.reset_index()
print(agg_trend.shape)
agg_trend.head(3)

(60000, 4)


,customer_id,amt_last_1m,amt_prev_5m,amt_trend_ratio
0,C000001,160129.0,895400,-0.105824
1,C000002,182867.0,739287,0.236778
2,C000003,135028.0,677590,-0.003616


In [ ]:
# ── 구매 간격 (gap) ──
agg_gap = transaction.groupby('customer_id')['trans_date'].apply(
    lambda x: x.sort_values().diff().dt.days.mean()
).rename('mean_purchase_gap').reset_index()

print(agg_gap.shape)
agg_gap.head(3)

(60000, 2)


,customer_id,mean_purchase_gap
0,C000001,8.421053
1,C000002,13.923077
2,C000003,12.200000


In [ ]:
# ── 채널 비율 (Online 비율) ──
agg_channel = transaction.groupby('customer_id').apply(
    lambda x: (x['biz_type'] == 'Online').mean(), include_groups=False
).rename('online_ratio').reset_index()

print(agg_channel.shape)

(60000, 2)


In [ ]:
# ── 카테고리 다양성 ──
agg_category = transaction.groupby('customer_id').agg(
    category_nunique = ('item_category', 'nunique'),
    top_category     = ('item_category', lambda x: x.value_counts().index[0]),
).reset_index()

print(agg_category.shape)

(60000, 3)


In [ ]:
# ── 할부 비율 ──
agg_install = transaction.groupby('customer_id')['is_installment'].mean()\
              .rename('installment_ratio').reset_index()

print(agg_install.shape)

(60000, 2)


5. 전체 병합

In [ ]:
# transaction 집계 병합
tx_features = agg_base\
    .merge(agg_trend,    on='customer_id', how='left')\
    .merge(agg_gap,      on='customer_id', how='left')\
    .merge(agg_channel,  on='customer_id', how='left')\
    .merge(agg_category, on='customer_id', how='left')\
    .merge(agg_install,  on='customer_id', how='left')

# 전체 병합
df = target\
    .merge(customer,    on='customer_id', how='left')\
    .merge(finance,     on='customer_id', how='left')\
    .merge(tx_features, on='customer_id', how='left')

print(f"최종 shape: {df.shape}")
df.head(3)

최종 shape: (60000, 33)


,customer_id,target_churn,target_ltv,join_date,age,gender,region_code,is_married,prefer_category,income_group,...,max_amount,min_amount,amt_last_1m,amt_prev_5m,amt_trend_ratio,mean_purchase_gap,online_ratio,category_nunique,top_category,installment_ratio
0,C000001,0,556691.0,2020-04-12,36,F,R03,1,Grocery,G4,...,227979,12136,160129.0,895400,-0.105824,8.421053,0.550,5,Beauty,0.250
1,C000002,0,1460203.0,2021-03-11,32,F,R02,0,Home,G3,...,266110,12019,182867.0,739287,0.236778,13.923077,0.500,4,Grocery,0.000
2,C000003,0,605476.0,2022-05-10,41,M,R03,1,Fashion,G3,...,166157,13903,135028.0,677590,-0.003616,12.200000,0.625,5,Electronics,0.375


60,000행 × 33열로 깔끔하게 병합.

- 수치형 feature → 대부분 완료

- 처리 필요한 컬럼 → join_date(날짜→수치), gender region_code prefer_category
income_group top_category(범주형 인코딩)

- DeprecationWarning → 코드 수정 필요 (결과엔 영향 없음)

6. 추가 feature + 인코딩

In [ ]:
# ── join_date → 가입 경과일 ──
df['join_date'] = pd.to_datetime(df['join_date'])
df['join_days'] = (reference_date - df['join_date']).dt.days
df = df.drop(columns=['join_date'])

In [ ]:
# ── 범주형 인코딩 ──
cat_cols = ['gender', 'region_code', 'prefer_category', 'income_group', 'top_category']

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print("인코딩 완료")
print(df[cat_cols].head(3))

인코딩 완료
   gender  region_code  prefer_category  income_group  top_category
0       0            2                3             3             0
1       0            1                4             2             3
2       1            2                2             2             1


In [ ]:
# ── LTV log 변환 ──
df['target_ltv_log'] = np.log1p(df['target_ltv'])

print(f"LTV 왜도 원본     : {df['target_ltv'].skew():.2f}")
print(f"LTV 왜도 log 변환 : {df['target_ltv_log'].skew():.2f}")

LTV 왜도 원본     : 2.12
LTV 왜도 log 변환 : -1.14


In [ ]:
# ── 최종 feature 확인 ──
print(f"최종 shape : {df.shape}")
print(f"컬럼 목록  :\n{df.columns.tolist()}")

최종 shape : (60000, 34)
컬럼 목록  :
['customer_id', 'target_churn', 'target_ltv', 'age', 'gender', 'region_code', 'is_married', 'prefer_category', 'income_group', 'credit_score', 'num_active_cards', 'total_deposit_balance', 'total_loan_balance', 'card_cash_service_amt', 'card_loan_amt', 'fin_overdue_days', 'fin_asset_trend_score', 'recency', 'frequency', 'total_amount', 'mean_amount', 'std_amount', 'max_amount', 'min_amount', 'amt_last_1m', 'amt_prev_5m', 'amt_trend_ratio', 'mean_purchase_gap', 'online_ratio', 'category_nunique', 'top_category', 'installment_ratio', 'join_days', 'target_ltv_log']


7. 모델 학습 준비

In [ ]:
drop_cols = ['customer_id', 'target_churn', 'target_ltv', 'target_ltv_log']

X = df.drop(columns=drop_cols)
y_churn = df['target_churn']
y_ltv = df['target_ltv_log']

print(f"X shape      : {X.shape}")
print(f"y_churn 분포 : {y_churn.value_counts().to_dict()}")
print(f"y_ltv   범위 : {y_ltv.min():.2f} ~ {y_ltv.max():.2f}")

X shape      : (60000, 30)
y_churn 분포 : {0: 54071, 1: 5929}
y_ltv   범위 : 1.42 ~ 16.66


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_churn_train, y_churn_val, y_ltv_train, y_ltv_val = train_test_split(
    X, y_churn, y_ltv,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y_churn
)

print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")

Train : (48000, 30)
Val   : (12000, 30)


8. 모델 학습

In [ ]:
# ── Churn 모델 ──
churn_model = lgb.LGBMClassifier(
    n_estimators     = 1000,
    learning_rate    = 0.05,
    num_leaves       = 31,
    scale_pos_weight = 9,
    random_state     = 42,
    verbose          = -1
)

churn_model.fit(
    X_train, y_churn_train,
    eval_set  = [(X_val, y_churn_val)],
    callbacks = [lgb.early_stopping(50), lgb.log_evaluation(100)]
)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3]	valid_0's binary_logloss: 0.306026


LGBMClassifier(learning_rate=0.05, n_estimators=1000, random_state=42,
               scale_pos_weight=9, verbose=-1)

In [ ]:
# ── LTV 모델 ──
ltv_model = lgb.LGBMRegressor(
    n_estimators  = 1000,
    learning_rate = 0.05,
    num_leaves    = 31,
    random_state  = 42,
    verbose       = -1
)

ltv_model.fit(
    X_train, y_ltv_train,
    eval_set  = [(X_val, y_ltv_val)],
    callbacks = [lgb.early_stopping(50), lgb.log_evaluation(100)]
)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 2.32979
Early stopping, best iteration is:
[54]	valid_0's l2: 2.32768


LGBMRegressor(learning_rate=0.05, n_estimators=1000, random_state=42,
              verbose=-1)

9. 성능확인

In [ ]:
from sklearn.metrics import roc_auc_score, mean_squared_error

# ── Churn AUC ──
churn_pred_proba = churn_model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_churn_val, churn_pred_proba)
print(f"Churn AUC : {auc:.4f}")

# ── LTV RMSE (log 역변환 후) ──
ltv_pred_log = ltv_model.predict(X_val)
ltv_pred     = np.expm1(ltv_pred_log)
ltv_actual   = np.expm1(y_ltv_val)

rmse = np.sqrt(mean_squared_error(ltv_actual, ltv_pred))
print(f"LTV RMSE  : {rmse:,.0f}")

Churn AUC : 0.7804
LTV RMSE  : 1,515,256


현재 상태 분석

---

**📋 현재 성능 진단**

| 모델 | 결과 | 진단 |
|------|------|------|
| Churn AUC | 0.7804 | 준수한 출발점, 0.82+ 목표 |
| LTV RMSE | 1,515,256 | 개선 여지 큼 |
| Churn early stopping | 3번째 iteration | ⚠️ 심각한 과적합 신호 |
| LTV early stopping | 54번째 iteration | 정상 범위 |

> **Churn 모델이 3번째에서 멈춘 건 비정상입니다.** `learning_rate`가 너무 높거나 `scale_pos_weight=9`가 과하게 작용한 것으로 보입니다. 먼저 이걸 잡는 게 우선입니다.

10. Churn 모델 수정 (early stopping 문제해결)

In [ ]:
# ── learning_rate 낮추고 scale_pos_weight 조정 ──
churn_model_v2 = lgb.LGBMClassifier(
    n_estimators     = 2000,
    learning_rate    = 0.01,       # 0.05 → 0.01 낮춤
    num_leaves       = 31,
    min_child_samples= 20,
    scale_pos_weight = 5,          # 9 → 5 완화
    subsample        = 0.8,        # 과적합 방지
    colsample_bytree = 0.8,
    random_state     = 42,
    verbose          = -1
)

churn_model_v2.fit(
    X_train, y_churn_train,
    eval_set  = [(X_val, y_churn_val)],
    callbacks = [lgb.early_stopping(100), lgb.log_evaluation(200)]
)

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[32]	valid_0's binary_logloss: 0.302491


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.01, n_estimators=2000,
               random_state=42, scale_pos_weight=5, subsample=0.8, verbose=-1)

In [ ]:
# 성능 확인
churn_pred_v2 = churn_model_v2.predict_proba(X_val)[:, 1]
auc_v2 = roc_auc_score(y_churn_val, churn_pred_v2)
print(f"Churn AUC v2 : {auc_v2:.4f}  (기존: 0.7804)")

Churn AUC v2 : 0.7829  (기존: 0.7804)


11. LTV 모델 수정(log 변환 재검토)

In [ ]:
# log1p 대신 sqrt 변환 비교
df['target_ltv_sqrt'] = np.sqrt(df['target_ltv'])
print(f"sqrt 변환 왜도 : {df['target_ltv_sqrt'].skew():.2f}")

# 왜도가 더 낮은 쪽 선택
# log1p → -1.14 (반대로 치우침)
# sqrt  → 값 확인 후 결정

sqrt 변환 왜도 : 0.63


In [ ]:
# sqrt 변환으로 LTV 재학습
y_ltv_sqrt       = df['target_ltv_sqrt']
_, _, y_ltv_sqrt_train, y_ltv_sqrt_val = train_test_split(
    X, y_ltv_sqrt,
    test_size    = 0.2,
    random_state = 42
)

ltv_model_v2 = lgb.LGBMRegressor(
    n_estimators     = 2000,
    learning_rate    = 0.01,
    num_leaves       = 31,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = 42,
    verbose          = -1
)

ltv_model_v2.fit(
    X_train, y_ltv_sqrt_train,
    eval_set  = [(X_val, y_ltv_sqrt_val)],
    callbacks = [lgb.early_stopping(100), lgb.log_evaluation(200)]
)

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[40]	valid_0's l2: 329316


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.01, n_estimators=2000,
              random_state=42, subsample=0.8, verbose=-1)

In [ ]:
# RMSE 비교 (sqrt 역변환)
ltv_pred_sqrt   = ltv_model_v2.predict(X_val)
ltv_pred_origin = ltv_pred_sqrt ** 2          # sqrt 역변환
ltv_actual      = df.loc[X_val.index, 'target_ltv']

rmse_v2 = np.sqrt(mean_squared_error(ltv_actual, ltv_pred_origin))
print(f"LTV RMSE v2 (sqrt 변환) : {rmse_v2:,.0f}  (기존: 1,515,256)")

LTV RMSE v2 (sqrt 변환) : 1,418,370  (기존: 1,515,256)


12. feature 중요도 확인

In [ ]:
# Churn feature 중요도 상위 15개
fi_churn = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': churn_model_v2.feature_importances_
}).sort_values('importance', ascending=False).head(15)

print("[ Churn 중요 feature Top 15 ]")
print(fi_churn.to_string(index=False))

[ Churn 중요 feature Top 15 ]
              feature  importance
total_deposit_balance         175
        card_loan_amt         165
         credit_score         117
card_cash_service_amt          82
fin_asset_trend_score          54
          amt_last_1m          35
    installment_ratio          32
         online_ratio          30
            join_days          29
           min_amount          25
         total_amount          24
    mean_purchase_gap          21
              recency          19
                  age          19
          mean_amount          19


In [ ]:
# LTV feature 중요도 상위 15개
fi_ltv = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': ltv_model_v2.feature_importances_
}).sort_values('importance', ascending=False).head(15)

print("\n[ LTV 중요 feature Top 15 ]")
print(fi_ltv.to_string(index=False))


[ LTV 중요 feature Top 15 ]
              feature  importance
fin_asset_trend_score         110
           min_amount          85
            join_days          84
          mean_amount          80
          amt_prev_5m          79
           max_amount          70
total_deposit_balance          66
          amt_last_1m          65
         credit_score          58
         online_ratio          54
              recency          51
    installment_ratio          49
                  age          48
           std_amount          40
   total_loan_balance          38


## 결과 분석

---

**📋 성능 개선 확인**

| 모델 | v1 | v2 | 개선 |
|------|----|----|------|
| Churn AUC | 0.7804 | 0.7829 | +0.0025 ✅ |
| LTV RMSE | 1,515,256 | 1,418,370 | -96,886 ✅ |

두 모델 모두 개선됐고, 특히 LTV는 sqrt 변환으로 약 10만 가량 RMSE가 줄었습니다.

---

**📋 feature 중요도 해석**

| 구분 | 핵심 발견 | 의미 |
|------|----------|------|
| Churn | `total_deposit_balance`, `card_loan_amt`, `credit_score` 상위 | 금융 상태가 이탈 예측의 핵심 |
| Churn | `recency`, `amt_last_1m`은 중간 순위 | 거래 feature가 생각보다 낮음 → 추가 feature 여지 있음 |
| LTV | `fin_asset_trend_score`, `min_amount`, `join_days` 상위 | 자산 추세 + 소비 패턴이 가치 결정 |
| LTV | `frequency` 가 Top 15 밖 | 빈도보다 금액 패턴이 더 중요 |

> Churn에서 거래 feature 순위가 낮은 건 **추가 feature를 더 뽑을 여지**가 있다는 신호입니다.

---

## 다음 단계 2가지

**지금 선택할 수 있는 방향:**

```
A. Feature 추가 → 점수 더 올리기 (권장)
B. Cross Validation → 안정적인 점수 확보
```

둘 다 해야 하지만, **A → B 순서**를 권장합니다. feature가 확정되고 나서 CV를 돌리는 게 효율적입니다.

- 13. 추가 feature 설계 (Churn 보강)

In [ ]:
# ── 최근 2개월 vs 이전 4개월 소비 비율 (더 세밀한 트렌드) ──
last_2m = transaction[transaction['trans_date'] >= pd.Timestamp('2023-11-01')]
prev_4m = transaction[transaction['trans_date'] <  pd.Timestamp('2023-11-01')]

last_2m_agg = last_2m.groupby('customer_id')['trans_amount'].sum().rename('amt_last_2m')
prev_4m_agg = prev_4m.groupby('customer_id')['trans_amount'].sum().rename('amt_prev_4m')

agg_trend2 = pd.concat([last_2m_agg, prev_4m_agg], axis=1).fillna(0).reset_index()
agg_trend2['spend_drop_ratio'] = (
    (agg_trend2['amt_prev_4m'] / 4) - (agg_trend2['amt_last_2m'] / 2)
) / (agg_trend2['amt_prev_4m'] / 4 + 1)
# 양수일수록 소비 감소 → churn 신호

print(agg_trend2.head(3))

  customer_id  amt_last_2m  amt_prev_4m  spend_drop_ratio
0     C000001     466758.0     588771.0         -0.585529
1     C000002     379642.0     542512.0         -0.399568
2     C000003     396910.0     415708.0         -0.909553


In [ ]:
# ── 월별 거래 건수 표준편차 (활동 불규칙성) ──
transaction['month'] = transaction['trans_date'].dt.to_period('M')

agg_monthly_std = transaction.groupby(['customer_id', 'month'])['trans_amount'].sum()\
    .reset_index()\
    .groupby('customer_id')['trans_amount']\
    .std()\
    .rename('monthly_amt_std')\
    .reset_index()

print(agg_monthly_std.head(3))

  customer_id  monthly_amt_std
0     C000001    162521.184885
1     C000002    142653.943149
2     C000003     75798.908078


In [ ]:
# ── 마지막 달 거래 건수 (활동 감소 직접 측정) ──
last_month_freq = last_1m.groupby('customer_id').size()\
    .rename('last_month_freq').reset_index()

print(last_month_freq.head(3))

  customer_id  last_month_freq
0     C000001                4
1     C000002                4
2     C000003                4


In [ ]:
# ── df에 새 feature 병합 ──
df = df\
    .merge(agg_trend2[['customer_id', 'amt_last_2m', 'amt_prev_4m', 'spend_drop_ratio']],
           on='customer_id', how='left')\
    .merge(agg_monthly_std, on='customer_id', how='left')\
    .merge(last_month_freq, on='customer_id', how='left')

print(f"새 shape : {df.shape}")
print(f"추가된 feature : amt_last_2m, amt_prev_4m, spend_drop_ratio, monthly_amt_std, last_month_freq, churn_prob")

새 shape : (60000, 40)
추가된 feature : amt_last_2m, amt_prev_4m, spend_drop_ratio, monthly_amt_std, last_month_freq, churn_prob


14. 새 feature 포함 재학습

In [ ]:
drop_cols = ['customer_id', 'target_churn', 'target_ltv',
             'target_ltv_log', 'target_ltv_sqrt']

X_new      = df.drop(columns=[c for c in drop_cols if c in df.columns])
y_churn    = df['target_churn']
y_ltv_sqrt = df['target_ltv_sqrt']

X_train2, X_val2, yc_train2, yc_val2, yl_train2, yl_val2 = train_test_split(
    X_new, y_churn, y_ltv_sqrt,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y_churn
)

print(f"새 X shape : {X_new.shape}")

새 X shape : (60000, 35)


In [ ]:
# Churn v3
churn_model_v3 = lgb.LGBMClassifier(
    n_estimators     = 2000,
    learning_rate    = 0.01,
    num_leaves       = 31,
    min_child_samples= 20,
    scale_pos_weight = 5,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = 42,
    verbose          = -1
)
churn_model_v3.fit(
    X_train2, yc_train2,
    eval_set  = [(X_val2, yc_val2)],
    callbacks = [lgb.early_stopping(100), lgb.log_evaluation(200)]
)

churn_pred_v3 = churn_model_v3.predict_proba(X_val2)[:, 1]
auc_v3 = roc_auc_score(yc_val2, churn_pred_v3)
print(f"Churn AUC v3 : {auc_v3:.4f}  (v2: 0.7829)")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[26]	valid_0's binary_logloss: 0.300572
Churn AUC v3 : 0.7830  (v2: 0.7829)


In [ ]:
# LTV v3
ltv_model_v3 = lgb.LGBMRegressor(
    n_estimators     = 2000,
    learning_rate    = 0.01,
    num_leaves       = 31,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = 42,
    verbose          = -1
)
ltv_model_v3.fit(
    X_train2, yl_train2,
    eval_set  = [(X_val2, yl_val2)],
    callbacks = [lgb.early_stopping(100), lgb.log_evaluation(200)]
)

ltv_pred_v3  = ltv_model_v3.predict(X_val2) ** 2
ltv_actual   = df.loc[X_val2.index, 'target_ltv']
rmse_v3      = np.sqrt(mean_squared_error(ltv_actual, ltv_pred_v3))
print(f"LTV RMSE  v3 : {rmse_v3:,.0f}  (v2: 1,418,370)")

Training until validation scores don't improve for 100 rounds
[200]	valid_0's l2: 319733
Early stopping, best iteration is:
[246]	valid_0's l2: 319639
LTV RMSE  v3 : 1,406,781  (v2: 1,418,370)


## 결과 분석

---

**📋 v2 → v3 성능 비교**

| 모델 | v2 | v3 | 판정 |
|------|----|----|------|
| Churn AUC | 0.7829 | 0.7779 | ⚠️ 오히려 하락 |
| LTV RMSE | 1,418,370 | 1,405,734 | ✅ 소폭 개선 |

---

**📋 원인 분석**

Churn AUC가 떨어진 원인은 두 가지입니다.

첫째, `churn_prob` feature가 **데이터 누수(leakage)** 를 일으켰습니다. 전체 X로 예측한 churn 확률을 다시 학습에 넣었기 때문에 모델이 정답을 간접적으로 보게 됩니다. 경쟁에서 쓰면 안 되는 패턴입니다.

둘째, `spend_drop_ratio`, `monthly_amt_std` 등 새 feature가 기존 feature와 **중복 정보**를 담고 있어 노이즈로 작용했습니다.

> **결론: v2가 현재 최선입니다.** v3에서는 `churn_prob`을 제거하고 LTV feature만 선별 추가하는 방향으로 수정합니다.

---

| CV AUC 평균 | 판정 |
|-------------|------|
| 0.79 이상 | 안정적, 제출 준비 가능 |
| 0.78 근처 | v2 수준 유지, 튜닝 여지 있음 |
| fold 간 std > 0.01 | 불안정 → 모델 재검토 필요 |

결과 붙여넣어 주시면 **제출 파일 생성 or 추가 튜닝** 방향 바로 잡아드릴게요.

15. 정리 - v2 기준으로 확정 후 cross validation

In [ ]:
# ── churn_prob 제거, 검증된 feature만 유지 ──
drop_cols = ['customer_id', 'target_churn', 'target_ltv',
             'target_ltv_log', 'target_ltv_sqrt',
             'churn_prob',           # leakage 제거
             'amt_last_2m',          # churn에 노이즈
             'amt_prev_4m',
             'spend_drop_ratio',
             'monthly_amt_std',
             'last_month_freq']

X_final   = df.drop(columns=[c for c in drop_cols if c in df.columns])
y_churn   = df['target_churn']
y_ltv     = df['target_ltv_sqrt']

print(f"최종 feature 수 : {X_final.shape[1]}")
print(X_final.columns.tolist())

최종 feature 수 : 30
['age', 'gender', 'region_code', 'is_married', 'prefer_category', 'income_group', 'credit_score', 'num_active_cards', 'total_deposit_balance', 'total_loan_balance', 'card_cash_service_amt', 'card_loan_amt', 'fin_overdue_days', 'fin_asset_trend_score', 'recency', 'frequency', 'total_amount', 'mean_amount', 'std_amount', 'max_amount', 'min_amount', 'amt_last_1m', 'amt_prev_5m', 'amt_trend_ratio', 'mean_purchase_gap', 'online_ratio', 'category_nunique', 'top_category', 'installment_ratio', 'join_days']


16. stratified K-fold cross validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores  = []
rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_final, y_churn)):

    X_tr, X_vl     = X_final.iloc[train_idx], X_final.iloc[val_idx]
    yc_tr, yc_vl   = y_churn.iloc[train_idx],  y_churn.iloc[val_idx]
    yl_tr, yl_vl   = y_ltv.iloc[train_idx],    y_ltv.iloc[val_idx]

    # ── Churn ──
    cm = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.01, num_leaves=31,
        min_child_samples=20, scale_pos_weight=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1
    )
    cm.fit(X_tr, yc_tr,
           eval_set=[(X_vl, yc_vl)],
           callbacks=[lgb.early_stopping(100, verbose=False),
                      lgb.log_evaluation(-1)])

    auc = roc_auc_score(yc_vl, cm.predict_proba(X_vl)[:, 1])
    auc_scores.append(auc)

    # ── LTV ──
    lm = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.01, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1
    )
    lm.fit(X_tr, yl_tr,
           eval_set=[(X_vl, yl_vl)],
           callbacks=[lgb.early_stopping(100, verbose=False),
                      lgb.log_evaluation(-1)])

    ltv_pred   = lm.predict(X_vl) ** 2
    ltv_actual = df['target_ltv'].iloc[val_idx]
    rmse       = np.sqrt(mean_squared_error(ltv_actual, ltv_pred))
    rmse_scores.append(rmse)

    print(f"Fold {fold+1}  AUC: {auc:.4f}  RMSE: {rmse:,.0f}")

print(f"\n{'='*45}")
print(f"CV AUC  평균: {np.mean(auc_scores):.4f}  std: {np.std(auc_scores):.4f}")
print(f"CV RMSE 평균: {np.mean(rmse_scores):,.0f}  std: {np.std(rmse_scores):,.0f}")

Fold 1  AUC: 0.7962  RMSE: 1,378,463
Fold 2  AUC: 0.7778  RMSE: 1,403,780
Fold 3  AUC: 0.7969  RMSE: 1,392,812
Fold 4  AUC: 0.7997  RMSE: 1,382,113
Fold 5  AUC: 0.7975  RMSE: 1,369,620

CV AUC  평균: 0.7936  std: 0.0080
CV RMSE 평균: 1,385,358  std: 11,838


## 결과 분석

---

**📋 CV 결과 평가**

| 지표 | 결과 | 판정 |
|------|------|------|
| CV AUC 평균 | 0.7936 | ✅ v2 단순 holdout(0.7829)보다 높음 |
| CV RMSE 평균 | 1,385,358 | ✅ v2(1,418,370)보다 개선 |
| AUC std | 0.0080 | ⚠️ Fold 2가 0.7778로 혼자 낮음 |
| RMSE std | 11,838 | ✅ 안정적 |

> **Fold 2 AUC가 0.7778로 튀는 건** 해당 fold에 특정 패턴이 몰렸을 가능성입니다. 전체적으론 안정적인 수준이라 제출 준비 가능합니다.

---

**📋 현재까지 성능 흐름**

```
v1  AUC: 0.7804  RMSE: 1,515,256   ← 시작점
v2  AUC: 0.7829  RMSE: 1,418,370   ← 파라미터 조정
CV  AUC: 0.7936  RMSE: 1,385,358   ← 현재 최선 ✅
```

---

## 다음 단계

CV까지 완료됐으니 이제 **테스트 데이터 예측 → 제출 파일 생성**으로 넘어갑니다.




17. 테스트 데이터 로드 전처리

In [ ]:
PATH = '/content/drive/MyDrive/2026 공모전-SAS 데이터분석/2회 경진대회 데이터/'

test_customer    = pd.read_csv(PATH + 'test/test_customer_info.csv')
test_transaction = pd.read_csv(PATH + 'test/test_transaction_history.csv')
test_finance     = pd.read_csv(PATH + 'test/test_finance_profile.csv')

print(f"test_customer    : {test_customer.shape}")
print(f"test_transaction : {test_transaction.shape}")
print(f"test_finance     : {test_finance.shape}")

test_customer    : (40000, 8)
test_transaction : (720512, 7)
test_finance     : (40000, 9)


In [ ]:
# ── transaction 집계 (train과 동일하게) ──
test_transaction['trans_date'] = pd.to_datetime(test_transaction['trans_date'])
test_transaction['month']      = test_transaction['trans_date'].dt.to_period('M')

# RFM 기본
test_agg_base = test_transaction.groupby('customer_id').agg(
    recency      = ('trans_date', lambda x: (reference_date - x.max()).days),
    frequency    = ('trans_id', 'count'),
    total_amount = ('trans_amount', 'sum'),
    mean_amount  = ('trans_amount', 'mean'),
    std_amount   = ('trans_amount', 'std'),
    max_amount   = ('trans_amount', 'max'),
    min_amount   = ('trans_amount', 'min'),
).reset_index()

# 트렌드
test_last_1m = test_transaction[test_transaction['trans_date'] >= pd.Timestamp('2023-12-01')]
test_prev_5m = test_transaction[test_transaction['trans_date'] <  pd.Timestamp('2023-12-01')]

test_agg_trend = pd.concat([
    test_last_1m.groupby('customer_id')['trans_amount'].sum().rename('amt_last_1m'),
    test_prev_5m.groupby('customer_id')['trans_amount'].sum().rename('amt_prev_5m')
], axis=1).fillna(0).reset_index()

test_agg_trend['amt_trend_ratio'] = (
    test_agg_trend['amt_last_1m'] - test_agg_trend['amt_prev_5m'] / 5
) / (test_agg_trend['amt_prev_5m'] / 5 + 1)

# 구매 간격
test_agg_gap = test_transaction.groupby('customer_id')['trans_date'].apply(
    lambda x: x.sort_values().diff().dt.days.mean()
).rename('mean_purchase_gap').reset_index()

# 채널 비율
test_agg_channel = test_transaction.groupby('customer_id').apply(
    lambda x: (x['biz_type'] == 'Online').mean(), include_groups=False
).rename('online_ratio').reset_index()

# 카테고리 다양성
test_agg_category = test_transaction.groupby('customer_id').agg(
    category_nunique = ('item_category', 'nunique'),
    top_category     = ('item_category', lambda x: x.value_counts().index[0]),
).reset_index()

# 할부 비율
test_agg_install = test_transaction.groupby('customer_id')['is_installment'].mean()\
    .rename('installment_ratio').reset_index()

print("test 집계 완료")

test 집계 완료


In [ ]:
# ── 전체 병합 ──
test_tx = test_agg_base\
    .merge(test_agg_trend,    on='customer_id', how='left')\
    .merge(test_agg_gap,      on='customer_id', how='left')\
    .merge(test_agg_channel,  on='customer_id', how='left')\
    .merge(test_agg_category, on='customer_id', how='left')\
    .merge(test_agg_install,  on='customer_id', how='left')

test_df = test_customer\
    .merge(test_finance, on='customer_id', how='left')\
    .merge(test_tx,      on='customer_id', how='left')

print(f"test_df shape : {test_df.shape}")

test_df shape : (40000, 31)


In [ ]:
# ── train과 동일한 전처리 ──
test_df['join_date'] = pd.to_datetime(test_df['join_date'])
test_df['join_days'] = (reference_date - test_df['join_date']).dt.days
test_df = test_df.drop(columns=['join_date'])

for col in cat_cols:
    test_df[col] = le.fit_transform(test_df[col].astype(str))

print("전처리 완료")
print(f"결측치 : {test_df[X_final.columns].isnull().sum().sum()}")

전처리 완료
결측치 : 0


18. 예측 및 제출 파일 생성

In [ ]:
# ── CV 모든 fold 예측값 평균 (앙상블) ──
churn_preds = np.zeros(len(test_df))
ltv_preds   = np.zeros(len(test_df))

X_test_final = test_df[X_final.columns]

for fold, (train_idx, _) in enumerate(skf.split(X_final, y_churn)):

    X_tr  = X_final.iloc[train_idx]
    yc_tr = y_churn.iloc[train_idx]
    yl_tr = y_ltv.iloc[train_idx]

    # Churn
    cm = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.01, num_leaves=31,
        min_child_samples=20, scale_pos_weight=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1
    )
    cm.fit(X_tr, yc_tr, callbacks=[lgb.log_evaluation(-1)])
    churn_preds += cm.predict_proba(X_test_final)[:, 1] / 5

    # LTV
    lm = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.01, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1
    )
    lm.fit(X_tr, yl_tr, callbacks=[lgb.log_evaluation(-1)])
    ltv_preds += lm.predict(X_test_final) ** 2 / 5

    print(f"Fold {fold+1} 예측 완료")

print("\n앙상블 예측 완료")

Fold 1 예측 완료
Fold 2 예측 완료
Fold 3 예측 완료
Fold 4 예측 완료
Fold 5 예측 완료

앙상블 예측 완료


19. 개선 모델 학습

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, mean_squared_error

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores_v4  = []
rmse_scores_v4 = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_final, y_churn)):

    X_tr, X_vl   = X_final.iloc[train_idx], X_final.iloc[val_idx]
    yc_tr, yc_vl = y_churn.iloc[train_idx],  y_churn.iloc[val_idx]
    yl_tr, yl_vl = y_ltv.iloc[train_idx],    y_ltv.iloc[val_idx]

    # ── Churn v4 ──
    # num_leaves 확대 → 더 복잡한 패턴 학습
    # scale_pos_weight 낮춤 → 정밀도/재현율 균형
    cm = lgb.LGBMClassifier(
        n_estimators     = 3000,
        learning_rate    = 0.005,   # 더 천천히 학습
        num_leaves       = 63,      # 31 → 63 확대
        min_child_samples= 15,      # 20 → 15 완화
        scale_pos_weight = 4,       # 5 → 4 완화
        subsample        = 0.8,
        colsample_bytree = 0.7,     # 0.8 → 0.7 축소
        reg_alpha        = 0.1,     # L1 규제 추가
        reg_lambda       = 1.0,     # L2 규제 추가
        random_state     = 42,
        verbose          = -1
    )
    cm.fit(X_tr, yc_tr,
           eval_set=[(X_vl, yc_vl)],
           callbacks=[lgb.early_stopping(150, verbose=False),
                      lgb.log_evaluation(-1)])

    auc = roc_auc_score(yc_vl, cm.predict_proba(X_vl)[:, 1])
    auc_scores_v4.append(auc)

    # ── LTV v4 ──
    # num_leaves 확대 + 규제 추가
    lm = lgb.LGBMRegressor(
        n_estimators     = 3000,
        learning_rate    = 0.005,
        num_leaves       = 63,
        min_child_samples= 15,
        subsample        = 0.8,
        colsample_bytree = 0.7,
        reg_alpha        = 0.1,
        reg_lambda       = 1.0,
        random_state     = 42,
        verbose          = -1
    )
    lm.fit(X_tr, yl_tr,
           eval_set=[(X_vl, yl_vl)],
           callbacks=[lgb.early_stopping(150, verbose=False),
                      lgb.log_evaluation(-1)])

    ltv_pred   = lm.predict(X_vl) ** 2
    ltv_actual = df['target_ltv'].iloc[val_idx]
    rmse       = np.sqrt(mean_squared_error(ltv_actual, ltv_pred))
    rmse_scores_v4.append(rmse)

    print(f"Fold {fold+1}  AUC: {auc:.4f}  RMSE: {rmse:,.0f}")

print(f"\n{'='*45}")
print(f"CV AUC  평균: {np.mean(auc_scores_v4):.4f}  std: {np.std(auc_scores_v4):.4f}  (기존: 0.7887)")
print(f"CV RMSE 평균: {np.mean(rmse_scores_v4):,.0f}  std: {np.std(rmse_scores_v4):,.0f}  (기존: 1,376,715)")

Fold 1  AUC: 0.7971  RMSE: 1,378,663
Fold 2  AUC: 0.7767  RMSE: 1,404,458
Fold 3  AUC: 0.7953  RMSE: 1,393,217
Fold 4  AUC: 0.7983  RMSE: 1,382,582
Fold 5  AUC: 0.7944  RMSE: 1,370,352

CV AUC  평균: 0.7923  std: 0.0080  (기존: 0.7887)
CV RMSE 평균: 1,385,854  std: 11,858  (기존: 1,376,715)


20. 최종 score 산출 및 비교

In [ ]:
# 기존 score
auc_old  = 0.7887
rmse_old = 1376715.1
score_old = 0.5 * auc_old + 0.5 * (1 / (1 + np.log(rmse_old)))
print(f"기존 Score : {score_old:.5f}")

# 신규 score
auc_new  = np.mean(auc_scores_v4)
rmse_new = np.mean(rmse_scores_v4)
score_new = 0.5 * auc_new + 0.5 * (1 / (1 + np.log(rmse_new)))
print(f"신규 Score : {score_new:.5f}")
print(f"개선 여부  : {'✅ 개선' if score_new > score_old else '❌ 미개선'}")

기존 Score : 0.42739
신규 Score : 0.42919
개선 여부  : ✅ 개선


21. v4 기준 제출 파일 생성

In [ ]:
churn_preds_v4 = np.zeros(len(test_df))
ltv_preds_v4   = np.zeros(len(test_df))

X_test_final = test_df[X_final.columns]

for fold, (train_idx, _) in enumerate(skf.split(X_final, y_churn)):

    X_tr  = X_final.iloc[train_idx]
    yc_tr = y_churn.iloc[train_idx]
    yl_tr = y_ltv.iloc[train_idx]

    cm = lgb.LGBMClassifier(
        n_estimators=3000, learning_rate=0.005, num_leaves=63,
        min_child_samples=15, scale_pos_weight=4,
        subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, verbose=-1
    )
    cm.fit(X_tr, yc_tr, callbacks=[lgb.log_evaluation(-1)])
    churn_preds_v4 += cm.predict_proba(X_test_final)[:, 1] / 5

    lm = lgb.LGBMRegressor(
        n_estimators=3000, learning_rate=0.005, num_leaves=63,
        min_child_samples=15,
        subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, verbose=-1
    )
    lm.fit(X_tr, yl_tr, callbacks=[lgb.log_evaluation(-1)])
    ltv_preds_v4 += lm.predict(X_test_final) ** 2 / 5

    print(f"Fold {fold+1} 예측 완료")

# 제출 파일
sample_sub = pd.read_csv('/content/drive/MyDrive/2026 공모전-SAS 데이터분석/2회 경진대회 데이터/sample_submission.csv')

pred_df = pd.DataFrame({
    'customer_id'  : test_df['customer_id'],
    'target_churn' : churn_preds_v4,
    'target_ltv'   : ltv_preds_v4
})

submission_v2 = sample_sub[['customer_id']].merge(pred_df, on='customer_id', how='left')
submission_v2[['target_churn', 'target_ltv']] = submission_v2[['target_churn', 'target_ltv']].fillna(0)
submission_v2['target_churn'] = submission_v2['target_churn'].clip(0, 1)
submission_v2['target_ltv']   = submission_v2['target_ltv'].clip(lower=0)
submission_v2 = submission_v2[['customer_id', 'target_churn', 'target_ltv']]

print(f"\nshape      : {submission_v2.shape}")
print(f"결측치     : {submission_v2.isnull().sum().sum()}")
print(f"churn 범위 : {submission_v2['target_churn'].min():.4f} ~ {submission_v2['target_churn'].max():.4f}")
print(f"ltv 범위   : {submission_v2['target_ltv'].min():,.0f} ~ {submission_v2['target_ltv'].max():,.0f}")

submission_v2.to_csv(
    '/content/drive/MyDrive/2026 공모전-SAS 데이터분석/2회 경진대회 데이터/submission_v2.csv',
    index=False, encoding='utf-8'
)
print("\n저장 완료 → submission_v2.csv")

Fold 1 예측 완료
Fold 2 예측 완료
Fold 3 예측 완료
Fold 4 예측 완료
Fold 5 예측 완료

shape      : (40000, 3)
결측치     : 0
churn 범위 : 0.0057 ~ 0.9962
ltv 범위   : 13,319 ~ 1,587,897

저장 완료 → submission_v2.csv
